In [4]:
import os
import csv
import threading
from datetime import datetime
import pandas as pd
import numpy as np
import torch
import mlflow
import mlflow.pytorch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, classification_report, f1_score
from flask import Flask, request, jsonify
from flask_cors import CORS

In [5]:
# ============================================================
# 1. KONFIGURASI PARAMETER & PATH
# ============================================================
CSV_PATH     = '02_dataset_klasifikasi_sentimen.csv'  # Pastikan file ini ada di direktori yang sama
MODEL_NAME   = 'ZiweiChen/FinBERT-FOMC'
BLOCK_SIZE   = 350
STEP_SIZE    = 300
FLASK_PORT   = 5001
MLFLOW_URI   = 'file:///D:/Collage/AnuJurnal/mlruns'
EXPERIMENT   = 'FOMC_Sentiment_FinBERT'
HISTORY_FILE = 'history_prediksi.csv'

# ============================================================
# 2. LOAD DATA & MODEL
# ============================================================
print("\n--- TAHAP 1: LOAD DATA & MODEL ---")
try:
    df = pd.read_csv(CSV_PATH, sep=';')
    if df.shape[1] < 2:
        df = pd.read_csv(CSV_PATH, sep=',')
except Exception:
    df = pd.read_csv(CSV_PATH)

df.columns = df.columns.str.strip()
df = df.dropna(subset=['Teks_Pidato', 'Sentimen_Aktual']).reset_index(drop=True)
print(f"✅ Data berhasil dimuat: {len(df)} baris dokumen The Fed.")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⏳ Memuat model {MODEL_NAME} ke {DEVICE}...")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model     = BertForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()
print("✅ Model berhasil dimuat.")

id2label = model.config.id2label
label2id = {v.lower(): k for k, v in id2label.items()}
IDX_HAWK = label2id.get('hawkish', 1)


--- TAHAP 1: LOAD DATA & MODEL ---
✅ Data berhasil dimuat: 48 baris dokumen The Fed.
⏳ Memuat model ZiweiChen/FinBERT-FOMC ke cpu...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ZiweiChen/FinBERT-FOMC
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model berhasil dimuat.


In [6]:
# ============================================================
# 3. FUNGSI INFERENSI (SLIDING WINDOW)
# ============================================================
def ekstrak_skor_hawkish(teks: str) -> float:
    if not isinstance(teks, str) or len(teks.strip()) == 0:
        return 0.0

    ids = tokenizer(teks, add_special_tokens=False, return_tensors='pt')['input_ids'][0]
    skor = []
    
    with torch.inference_mode():
        for i in range(0, max(1, len(ids)), STEP_SIZE):
            blok = ids[i : i + BLOCK_SIZE]
            blok = torch.cat([
                torch.tensor([tokenizer.cls_token_id]),
                blok,
                torch.tensor([tokenizer.sep_token_id])
            ])[:512].unsqueeze(0).to(DEVICE)

            logits = model(blok).logits
            probs  = torch.softmax(logits, dim=-1)
            skor.append(probs[0][IDX_HAWK].item())

    return float(max(skor)) if skor else 0.0

def cari_threshold_optimal(skor_arr: np.ndarray, label_arr: np.ndarray, n_grid: int = 200) -> float:
    y = np.where(label_arr == 'Positif (Dovish)', 1, 0)
    grid = np.linspace(np.percentile(skor_arr, 5), np.percentile(skor_arr, 95), n_grid)
    loo = LeaveOneOut()
    results = []
    
    for t in grid:
        preds = np.array([(0 if skor_arr[ti[0]] >= t else 1) for _, ti in loo.split(skor_arr)])
        results.append((t, np.mean(preds == y)))

    arr = np.array(results)
    best_acc = arr[:, 1].max()
    return float(arr[arr[:, 1] == best_acc, 0].mean())

In [ ]:
# ============================================================
# 4. TRACKING & LOGGING MLFLOW
# ============================================================
print("\n--- TAHAP 2: MLFLOW EXPERIMENT ---")
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)

print("⏳ Menjalankan inferensi seluruh dataset untuk mencari threshold...")
with mlflow.start_run(run_name='FinBERT_FOMC_Production') as run:
    
    # A. Log Hyperparameters
    mlflow.log_params({
        'model_name': MODEL_NAME,
        'block_size': BLOCK_SIZE,
        'step_size': STEP_SIZE,
        'cv_strategy': 'LOO-CV'
    })

    # B. Hitung Skor & Threshold
    df['Hawkish_Score'] = df['Teks_Pidato'].apply(ekstrak_skor_hawkish)
    global thr_opt
    thr_opt = cari_threshold_optimal(df['Hawkish_Score'].values, df['Sentimen_Aktual'].values)
    df['Prediksi'] = np.where(df['Hawkish_Score'] >= thr_opt, 'Negatif (Hawkish)', 'Positif (Dovish)')

    # C. Hitung Metrik & Log
    acc = accuracy_score(df['Sentimen_Aktual'], df['Prediksi'])
    f1_h = f1_score(df['Sentimen_Aktual'], df['Prediksi'], pos_label='Negatif (Hawkish)', zero_division=0)
    
    mlflow.log_metrics({
        'accuracy': round(acc, 4), 
        'threshold_opt': round(thr_opt, 6), 
        'f1_hawkish': round(f1_h, 4)
    })

    # D. Simpan Artifacts
    with open('threshold.txt', 'w') as f_thr: 
        f_thr.write(str(thr_opt))
    mlflow.log_artifact('threshold.txt')
    
    output_csv = '03_hasil_finbert_klasifikasi_.csv'
    df.to_csv(output_csv, sep=';', index=False, encoding='utf-8-sig')
    mlflow.log_artifact(output_csv)
    
    print(f"✅ Eksperimen MLflow Selesai! (Run ID: {run.info.run_id})")
    print(f"🎯 Threshold Optimal: {thr_opt:.4f} | Akurasi: {acc:.2%}")

# ============================================================
# 5. DEPLOYMENT REST API FLASK + HISTORY LOGGING
# ============================================================
print("\n--- TAHAP 3: DEPLOYMENT FLASK API ---")
app = Flask(__name__)
CORS(app)

# Inisialisasi file history CSV jika belum ada (Bikin Header)
if not os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(['Waktu', 'Teks_Cuplikan', 'Skor_Hawkish', 'Net_Sentiment', 'Label_Prediksi', 'Status_Pasar'])

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'Online', 'model': MODEL_NAME, 'threshold_aktif': round(thr_opt, 4)})

@app.route('/analyze', methods=['POST'])
def analyze_text():
    try:
        data = request.get_json(force=True, silent=True)
        if not data or 'text' not in data:
            return jsonify({'error': 'Body JSON harus memiliki field "text"'}), 400
            
        text = data['text'].strip()
        if not text: return jsonify({'error': 'Teks kosong'}), 400

        # Eksekusi Model
        peak_hawkish = ekstrak_skor_hawkish(text)
        is_hawkish   = peak_hawkish >= thr_opt

        # Format Logika UI
        if is_hawkish:
            label          = 'Negatif (Hawkish)'
            bar_type       = 'hawkish'
            net_sentiment  = -round(peak_hawkish, 4)
            bar_pct        = round(min(100.0, peak_hawkish * 100), 2)
            summary        = 'Terdapat nada pengetatan moneter (Hawkish) yang kuat dari The Fed.'
            btc_reasoning  = 'Aset berisiko seperti Bitcoin cenderung mengalami tekanan jual (Bearish).'
            status_pasar   = 'BEARISH'
        else:
            label          = 'Positif (Dovish)'
            bar_type       = 'dovish'
            net_sentiment  = round(1.0 - peak_hawkish, 4)
            bar_pct        = round(min(100.0, (1.0 - peak_hawkish) * 100), 2)
            summary        = 'Dokumen The Fed didominasi nada pelonggaran (Dovish).'
            btc_reasoning  = 'Investor terdorong ke aset berisiko seperti Bitcoin (Risk-On / Bullish).'
            status_pasar   = 'BULLISH'

        # Ekstrak 20 kata pertama untuk cuplikan history
        snippet = ' '.join(text.split()[:20]) + '...'

        # --- SIMPAN KE FILE HISTORY ---
        waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(HISTORY_FILE, mode='a', newline='', encoding='utf-8') as file_csv:
            writer = csv.writer(file_csv, delimiter=';')
            writer.writerow([waktu_sekarang, snippet, round(peak_hawkish, 4), net_sentiment, label, status_pasar])

        return jsonify({
            'label'         : label,
            'hawkish_score' : round(peak_hawkish, 4),
            'threshold'     : round(thr_opt, 4),
            'net_sentiment' : net_sentiment,
            'bar_type'      : bar_type,
            'bar_pct'       : bar_pct,
            'summary'       : summary,
            'btc_reasoning' : btc_reasoning
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Menjalankan server Flask di background thread (penting jika dijalankan di Jupyter)
flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=FLASK_PORT, use_reloader=False, debug=False),
    daemon=True
)
flask_thread.start()

print(f"🚀 Server Flask API Berjalan di: http://localhost:{FLASK_PORT}")
print(f"   History prediksi web otomatis tersimpan di: {HISTORY_FILE}")
print("   API Endpoint : POST http://localhost:5001/analyze")


--- TAHAP 2: MLFLOW EXPERIMENT ---
⏳ Menjalankan inferensi seluruh dataset untuk mencari threshold...


C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


✅ Eksperimen MLflow Selesai! (Run ID: 65978203925c43a2b6171ab31f8570ba)
🎯 Threshold Optimal: 0.9105 | Akurasi: 64.58%

--- TAHAP 3: DEPLOYMENT FLASK API ---
🚀 Server Flask API Berjalan di: http://localhost:5001
   History prediksi web otomatis tersimpan di: history_prediksi.csv
   API Endpoint : POST http://localhost:5001/analyze


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://192.168.1.2:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/May/2026 20:41:17] "OPTIONS /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/May/2026 20:42:56] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/May/2026 20:45:32] "OPTIONS /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/May/2026 20:45:32] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/May/2026 20:47:00] "OPTIONS /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [15/May/2026 20:47:00] "POST /analyze HTTP/1.1" 200 -
